## Access to DP2

- author : Sylvie Dagoret-Campagne
- creation date : 2026-03-07

## Imports

In [ ]:
import lsst.pipe.base

print(lsst.pipe.base.__version__)

In [ ]:
import sys
import matplotlib.pyplot as plt
import lsst.afw.display as afwDisplay
from lsst.geom import SpherePoint, degrees
from lsst.afw.image import ExposureF
from lsst.skymap import PatchInfo, Index2D
import numpy as np
import pandas as pd
from astropy.time import Time
# %matplotlib widget
import seaborn as sns

In [ ]:
# Define butler
from lsst.daf.butler import Butler

In [ ]:
def get_uris_from_butler(butler, datasetType, collections=None, **query_kwargs):
    """
    Return a list of URIs for a given datasetType in a Butler collection.

    Parameters
    ----------
    butler : lsst.daf.butler.Butler
        The Butler object
    datasetType : str
        The dataset type, e.g. 'raw', 'calexp'
    collections : list of str, optional
        Butler collections to query
    query_kwargs : dict
        Additional query parameters, e.g. visit=123, filter='r'

    Returns
    -------
    uris : list of str
        File paths of the datasets
    """
    refs = butler.registry.queryDatasets(datasetType, collections=collections, **query_kwargs)
    return [butler.getURI(ref) for ref in refs]

In [ ]:
def get_refs_from_butler(butler, datasetType, collections=None, **query_kwargs):
    """
    Return a list of Butler DatasetRef objects for a given datasetType in a Butler collection.

    Parameters
    ----------
    butler : lsst.daf.butler.Butler
        The Butler object
    datasetType : str
        The dataset type, e.g. 'raw', 'calexp'
    collections : list of str, optional
        Butler collections to query
    query_kwargs : dict
        Additional query parameters, e.g. visit=123, filter='r'

    Returns
    -------
    refs : list of DatasetRef
        Butler dataset references
    """
    refs = butler.registry.queryDatasets(datasetType, collections=collections, **query_kwargs)
    return refs

In [ ]:
def get_refs_from_butler(
    butler,
    datasetType,
    collections=None,
    return_uris=False,
    **query_kwargs
):
    """
    Get dataset references (or URIs) from a Butler, safely handling numpy types.

    Parameters
    ----------
    butler : lsst.daf.butler.Butler
        Butler instance.
    datasetType : str
        Dataset type to query (e.g., "calexp", "raw", "src").
    collections : str or list of str, optional
        Butler collections to query.
    return_uris : bool, default False
        If True, return URIs instead of refs.
    **query_kwargs :
        Query parameters, e.g., visit=[...], ccd=[...], filter=['r','g'].

    Returns
    -------
    list of DatasetRef or list of str
        List of references or URIs.
    """

    # Convert numpy types to Python native types
    safe_kwargs = {}
    for k, v in query_kwargs.items():
        if isinstance(v, np.ndarray):
            safe_kwargs[k] = [int(x) if np.issubdtype(type(x), np.integer) else x for x in v]
        elif isinstance(v, (np.integer, np.int64, np.int32)):
            safe_kwargs[k] = int(v)
        else:
            safe_kwargs[k] = v

    # Query dataset references
    refs = butler.registry.queryDatasets(
        datasetType,
        collections=collections,
        **safe_kwargs
    )

    if return_uris:
        return [butler.getURI(ref) for ref in refs]

    return refs

## Configuration

- See collection here : https://usdf-rsp.slac.stanford.edu/plot-navigator
- See Campaign : https://rubinobs.atlassian.net/wiki/spaces/DM/pages/661192727/LSSTCam+Intermittent+DRP+Runs

In [ ]:
REPO = 'dp2_prep'
#collection = 'LSSTCam/runs/DRP/v30_0_1/DM-54061'
collection =  'LSSTCam/runs/DRP/v30_0_4/DM-54249'
#collection =  'LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881'
skymapName = "lsst_cells_v2"
instrument = "LSSTCam"
date_start = 20260101
date_selection = 20260310
where_clause = "instrument = '" + f"{instrument}" + "'"
where_clause_date = where_clause + f"and day_obs >= {date_start}"
butler = Butler(REPO, collections=collection)
registry = butler.registry

In [ ]:
try:
    skymap = butler.get("skyMap", skymap=skymapName, collections=collection)
except Exception as inst:
    print(type(inst))  # the exception type
    print(inst.args)  # arguments stored in .args
    print(inst)  # __str__ allows args to be printed directly,

In [ ]:
skymap

In [ ]:
df_exposure = pd.DataFrame(
    {
        "id": pd.Series(dtype="int"),
        "obs_id": pd.Series(dtype="int"),
        "day_obs": pd.Series(dtype="int"),
        "seq_num": pd.Series(dtype="int"),
        "time_start": pd.Series(dtype="str"),  # ou 'datetime64[ns]' si c’est un datetime
        "time_end": pd.Series(dtype="str"),  # idem
        "type": pd.Series(dtype="str"),
        "target": pd.Series(dtype="str"),
        "filter": pd.Series(dtype="str"),
        "zenith_angle": pd.Series(dtype="float"),
        "expos": pd.Series(dtype="float"),  # ou 'int' selon le cas
        "ra": pd.Series(dtype="float"),
        "dec": pd.Series(dtype="float"),
        "skyangle": pd.Series(dtype="float"),
        "azimuth": pd.Series(dtype="float"),
        "zenith": pd.Series(dtype="float"),
        "science_program": pd.Series(dtype="str"),
        "jd": pd.Series(dtype="float"),
        "mjd": pd.Series(dtype="float"),
    }
)

In [ ]:
# save the data array in rows before saving in pandas dataframe
rows = []
for count, info in enumerate(registry.queryDimensionRecords("exposure", where=where_clause_date)):
    try:
        jd_start = info.timespan.begin.value
        jd_end = info.timespan.end.value
        the_Time_start = Time(jd_start, format="jd", scale="utc")
        the_Time_end = Time(jd_end, format="jd", scale="utc")
        mjd_start = the_Time_start.mjd
        mjd_end = the_Time_end.mjd
        isot_start = the_Time_start.isot
        isot_end = the_Time_end.isot

        if count == 0:
            print("===== Time Conversion Debug Info =====")
            print(f"JD start      : {jd_start} (type: {type(jd_start)})")
            print(f"JD end        : {jd_end} (type: {type(jd_end)})")
            print(f"MJD start     : {mjd_start} (type: {type(mjd_start)})")
            print(f"MJD end       : {mjd_end} (type: {type(mjd_end)})")
            print(f"ISOT start    : {isot_start} (type: {type(isot_start)})")
            print(f"ISOT end      : {isot_end} (type: {type(isot_end)})")
            print("=======================================")

        # put row in a dictionnary before stacking
        row = {
            "id": info.id,
            "obs_id": info.obs_id,
            "day_obs": info.day_obs,
            "seq_num": info.seq_num,
            "time_start": isot_start,
            "time_end": isot_end,
            "type": info.observation_type,
            "target": info.target_name,
            "filter": info.physical_filter,
            "zenith_angle": info.zenith_angle,
            "expos": info.exposure_time,  # Exemple : adapter selon ton objet
            "ra": info.tracking_ra,
            "dec": info.tracking_dec,
            "skyangle": info.sky_angle,
            "azimuth": info.azimuth,
            "zenith": info.zenith_angle,
            "science_program": info.science_program,
            "jd": float(jd_start),
            "mjd": float(mjd_start),
        }
        rows.append(row)
        if count>50000:
            break

    except ValueError as e:
        print(f"Erreur de valeur : {e}")
    except FileNotFoundError as e:
        print(f"Fichier introuvable : {e}")
    except Exception as e:
        print(f"Erreur inattendue : {type(e).__name__} - {e}")
        print(f">>>   Unexpected error at row {count}:", sys.exc_info()[0])
        traceback.print_exc()  # affiche la stack trace complète

In [ ]:
df_exposure = pd.DataFrame(rows)

In [ ]:
df_science = df_exposure[df_exposure.type == "science"]
df_science.reset_index(drop=True, inplace=True)

In [ ]:
df_science.target.unique()

In [ ]:
def get_tract_patch(row, skymap):
    if pd.isna(row['ra']) or pd.isna(row['dec']):
        return pd.Series({"tract": None, "patch": None})
    
    target_point = SpherePoint(row['ra'], row['dec'], degrees)

    tract_info = skymap.findTract(target_point)
    patch_info = tract_info.findPatch(target_point)
    tractNbSel = tract_info.getId()
    patchNbSel =  patch_info.getSequentialIndex()
    patch_index_str = f"{patch_info.getIndex()[0]},{patch_info.getIndex()[1]}"
   
    
    return pd.Series({"tract":   tractNbSel, "patch":  patchNbSel, "patch_str": patch_index_str})


In [ ]:
df=df_science.copy()
df["band"] = df["filter"].apply(lambda x : x.split("_")[0])

In [ ]:
df[['tract', 'patch', "patch_str"]] = df.apply(get_tract_patch, axis=1, args=(skymap,))

In [ ]:
df

In [ ]:
print(list(df.target.unique()))

In [ ]:
df_cosmos = df[ (df.target ==  'ddf_cosmos, lowdust') | (df.target ==  '9813_lowdust, ddf_cosmos') ]
df_elaiss1 = df[df.target ==  'ddf_elaiss1, lowdust' ]

In [ ]:
df_cosmos

In [ ]:
df_elaiss1 

In [ ]:
# 1. Trier le DataFrame par numéro de tract
df = df.sort_values("tract")
# 2. Créer la colonne tag après le tri
df["tag"] = df["tract"].astype(str) + "_" + df["target"]
# 3. Grouper par tag et band, et compter
grouped_tag = df.groupby(['tag', 'band']).size().reset_index(name='count')
# 4. Définir l'ordre des tags selon l'ordre dans df trié
tag_order = df["tag"].drop_duplicates().tolist()

In [ ]:
# Forcer l'ordre des bandes
band_order = ['u', 'g', 'r', 'i', 'z', 'y']
color_map = {
    'u': 'blue',
    'g': 'green',
    'r': 'red',
    'i': 'orange',
    'z': 'gray',
    'y': 'black'
}

In [ ]:
grouped_tag

In [ ]:
List_Of_fields = list(grouped_tag["tag"].unique()) 

In [ ]:
for field in List_Of_fields:
    if "ddf" in field:
        print(field)

In [ ]:
# Pivot pour avoir chaque bande en colonne
df_wide = grouped_tag.pivot(index='tag', columns='band', values='count').fillna(0)

# Ordonner les bandes
band_order = [ 'g', 'r', 'i', 'z', 'y']
df_wide = df_wide[band_order]
# Couleurs par bande
color_map = {
    'u': '#0000ff',  # bleu
    'g': '#00ff00',  # vert
    'r': '#ff0000',  # rouge
    'i': '#ffaa00',  # jaune-orangé
    'z': '#7f00ff',  # violet froid
    'y': '#00cccc'   # cyan froid
}


In [ ]:
plt.figure(figsize=(20, 8))

bottom = pd.Series([0]*len(df_wide), index=df_wide.index)

for band in band_order:
    plt.bar(
        df_wide.index,
        df_wide[band],
        bottom=bottom,
        color=color_map[band],
        label=band,width=1
    )
    bottom += df_wide[band]

plt.xlabel("Visited fields / tract", fontsize=14)
plt.ylabel("Number of visits per band", fontsize=14)
plt.title("Stacked visits per band in each tract", fontsize=20, fontweight="bold")
plt.xticks(rotation=45, ha='right')
plt.legend(title="Band")
plt.tight_layout()
plt.show()

In [ ]:

#plt.figure(figsize=(12, max(6, len(df_wide)//2)))  # ajuster la hauteur selon le nombre de tracts
plt.figure(figsize=(16,80))  # ajuster la hauteur selon le nombre de tracts

left = pd.Series([0]*len(df_wide), index=df_wide.index)  # accumulation pour l'empilement

for band in band_order:
    plt.barh(
        y=df_wide.index,
        width=df_wide[band],
        left=left,
        color=color_map[band],
        label=band,
        height=1.0
    )
    left += df_wide[band]

plt.ylabel("Visited fields / tract", fontsize=14)
plt.xlabel("Number of visits per band", fontsize=14)
plt.title("Stacked visits per band in each tract (horizontal)", fontsize=20, fontweight="bold")
plt.legend(title="Band", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.ylim(-0.2, len(df_wide)-0.2)
plt.tight_layout()
plt.show()

In [ ]:
selected_tags = [
    "2233_ddf_edfs_a, lowdust",
    "2234_ddf_edfs_a, lowdust",
    "2234_lowdust, ddf_edfs_a",
    "2394_ddf_edfs_a, lowdust",
    "2394_lowdust, ddf_edfs_a",
    "2395_ddf_edfs_b, lowdust",
    "2396_ddf_edfs_b, lowdust",
    "2561_ddf_edfs_b, lowdust",
    "2876_ddf_elaiss1, lowdust",
    "2877_ddf_elaiss1, lowdust",
    "4848_ddf_ecdfs, lowdust",
    "4848_lowdust, ddf_ecdfs",
    "4849_lowdust, ddf_ecdfs",
    "5063_ddf_ecdfs, lowdust",
    "5063_lowdust, ddf_ecdfs",
    "8524_ddf_xmm_lss, lowdust",
    "8524_lowdust, ddf_xmm_lss",
    "8766_lowdust, ddf_xmm_lss",
    "9813_ddf_cosmos, lowdust",
    "9813_lowdust, ddf_cosmos"
]

In [ ]:
df_subset = grouped_tag[grouped_tag['tag'].isin(selected_tags)]

In [ ]:
df_wide = df_subset.pivot(index='tag', columns='band', values='count').fillna(0)
df_wide = df_wide[band_order]  # keep bands in order u→y

In [ ]:
plt.figure(figsize=(12, len(df_wide)*0.5))  # height proportional to number of tags
left = pd.Series([0]*len(df_wide), index=df_wide.index)

for band in band_order:
    plt.barh(
        y=df_wide.index,
        width=df_wide[band],
        left=left,
        color=color_map[band],
        label=band,
        height=0.6
    )
    left += df_wide[band]

plt.xlabel("Number of visits per band")
plt.ylabel("Tag / Tract")
plt.title("Stacked visits per band (selected tags)", fontsize=18, fontweight="bold")
plt.legend(title="Band", bbox_to_anchor=(1.05,1), loc='upper left')

# Remove empty space above/below
plt.ylim(-0.5, len(df_wide)-0.5)

plt.tight_layout()
plt.show()

In [ ]:
#butler.registry.queryDatasetTypes()

In [ ]:
FLAG_DUMP_DATASETS = True
if FLAG_DUMP_DATASETS:
    for datasetType in registry.queryDatasetTypes():
        if registry.queryDatasets(datasetType, collections=collection).any(
            execute=False, exact=False
        ):
            # Limit search results to the data products
            if (
                ("_config" not in datasetType.name)
                and ("_log" not in datasetType.name)
                and ("_metadata" not in datasetType.name)
                and ("_resource_usage" not in datasetType.name)
                and ("Plot" not in datasetType.name)
                and ("Metric" not in datasetType.name)
                and ("metric" not in datasetType.name)
            ):
                if "object" in datasetType.name or "Obj" in datasetType.name:
                    print(datasetType)
                if "source" in datasetType.name or "Source" in datasetType.name:
                    print(datasetType)
                else:
                    print(datasetType)

In [ ]:
visits = df_elaiss1["id"].values
visit_list = [int(v) for v in visits]

In [ ]:
visit_str = ",".join(str(v) for v in visit_list)

where = f"instrument=\'{instrument}\' AND visit IN ({visit_str})"
refs = list(
    butler.registry.queryDatasets(
        "raw",
        collections=collection,
        where=where
    )
)

In [ ]:
len(refs)

In [ ]:
print(butler.registry.getDatasetType("dia_source").dimensions)

In [ ]:
print(butler.registry.getDatasetType("dia_object").dimensions)